In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/CDS_Project"
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

print(f" Drive mounted. Output will be saved to: {DRIVE_OUTPUT_DIR}")

In [ ]:
!pip install pydriller pyyaml pandas tqdm --quiet
print(" Dependencies installed")

In [ ]:
import subprocess
import os

PROJECTKB_DIR = "/content/project-kb"

if os.path.isdir(os.path.join(PROJECTKB_DIR, ".git")):
    print(" ProjectKB already cloned — skipping")
else:
    print("Cloning ProjectKB (vulnerability-data branch)...")
    result = subprocess.run(
        [
            "git", "clone",
            "--depth", "1",                          # only latest snapshot
            "--branch", "vulnerability-data",        # the branch with YAML files
            "https://github.com/SAP/project-kb.git",
            PROJECTKB_DIR
        ],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(" ProjectKB cloned successfully")
    else:
        print(" Clone failed:", result.stderr)

STATEMENTS_DIR = os.path.join(PROJECTKB_DIR, "statements")

cve_dirs = [d for d in os.listdir(STATEMENTS_DIR)
            if os.path.isdir(os.path.join(STATEMENTS_DIR, d))]

print(f"\n Found {len(cve_dirs)} CVE folders in statements/")

In [ ]:
import yaml
import glob

cve_data = []   # will hold dicts: {cve, repo_url, commit_hash}

yaml_files = glob.glob(os.path.join(STATEMENTS_DIR, "*", "statement.yaml"))
print(f"Parsing {len(yaml_files)} YAML files...")

skipped_no_commit = 0

for yaml_path in yaml_files:
    with open(yaml_path, "r", encoding="utf-8", errors="ignore") as f:
        data = yaml.safe_load(f) or {}

    cve_id = data.get("vulnerability_id", "")

    fixes = data.get("fixes") or []
    if not fixes:
        skipped_no_commit += 1
        continue

    for fix in fixes:
        commits = fix.get("commits") or []
        for commit in commits:
            repo_url    = commit.get("repository", "").strip().rstrip("/")
            commit_hash = str(commit.get("id", "")).strip()

            if repo_url and commit_hash and len(commit_hash) >= 7:
                cve_data.append({
                    "cve":         cve_id,
                    "repo_url":    repo_url,
                    "commit_hash": commit_hash,
                })

print(f"\n Task 2 Results:")
print(f"   CVEs with ≥1 fixing commit : {len(yaml_files) - skipped_no_commit}")
print(f"   CVEs with NO commit (skipped): {skipped_no_commit}")
print(f"   Total (CVE, repo, commit) rows: {len(cve_data)}")

unique_repos = len(set(e["repo_url"] for e in cve_data))
print(f"   Unique repositories: {unique_repos}")

print(f"\nFirst 5 entries:")
for e in cve_data[:5]:
    print(f"  {e['cve']:<25} {e['commit_hash'][:10]}  {e['repo_url']}")

In [ ]:

CACHE_DIR        = "/content/repo_cache"  # where cloned repos are stored (Colab temp disk)
MAX_WORKERS      = 4                       # parallel clones (4 is safe for Colab)
NEG_SAMPLE_RATIO = 10                      # unchanged methods kept per changed pair
CLONE_DEPTH      = 500                     # how deep to clone (500 is enough for most repos)
CLONE_TIMEOUT    = 180                     # seconds before giving up on a slow repo

MAX_REPOS = None   # Change to a number if you want to limit


os.makedirs(CACHE_DIR, exist_ok=True)
print(f"Configuration:")
print(f"  Cache dir      : {CACHE_DIR}")
print(f"  Max workers    : {MAX_WORKERS}")
print(f"  Neg sample ratio: {NEG_SAMPLE_RATIO}")
print(f"  Clone depth    : {CLONE_DEPTH}")
print(f"  Max repos      : {'ALL' if MAX_REPOS is None else MAX_REPOS}")

In [ ]:
import subprocess
import random
import hashlib
import threading
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pydriller import Repository
from tqdm import tqdm

random.seed(42)


def repo_local_path(repo_url: str) -> str:
    slug = (repo_url.rstrip("/")
                    .replace("https://github.com/", "")
                    .replace("https://gitbox.apache.org/repos/asf/", "apache__")
                    .replace(".git", "")
                    .replace("/", "__"))
    return os.path.join(CACHE_DIR, slug)


def ensure_repo(repo_url: str):
    local = repo_local_path(repo_url)

    if os.path.isdir(os.path.join(local, ".git")):
        return local

    try:
        result = subprocess.run(
            [
                "git", "clone",
                "--depth", str(CLONE_DEPTH),  # shallow clone — don't need full history
                "--no-single-branch",          # fetch all branches (fixing commit may be on any)
                "--quiet",
                repo_url,
                local
            ],
            capture_output=True,
            text=True,
            timeout=CLONE_TIMEOUT,
            env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},  # don't hang waiting for password
        )
        return local if result.returncode == 0 else None
    except Exception:
        return None


def extract_method_code(source_lines: list, start: int, end: int) -> str:
    return "\n".join(source_lines[start - 1 : end])


def process_repo(repo_url: str, commit_hashes: list) -> tuple:
    local = ensure_repo(repo_url)
    if local is None:
        return [], {"clone_fail": 1}

    stats     = defaultdict(int)
    positives = []  # (code_str, True/False) for changed methods
    negatives = []  # (code_str, False) for unchanged methods

    try:
        for commit in Repository(local, only_commits=commit_hashes).traverse_commits():
            stats["commits_processed"] += 1

            for mf in commit.modified_files:
                if not mf.source_code_before or not mf.source_code:
                    stats["skipped_no_source"] += 1
                    continue

                if not mf.methods_before and not mf.methods:
                    stats["skipped_no_methods"] += 1
                    continue

                stats["files_processed"] += 1

                lines_before = mf.source_code_before.splitlines()
                lines_after  = mf.source_code.splitlines()

                before_map = {m.name: m for m in mf.methods_before}
                after_map  = {m.name: m for m in mf.methods}

                for name, mb in before_map.items():
                    ma = after_map.get(name)

                    if ma is None:
                        continue

                    stats["methods_compared"] += 1

                    vuln_code  = extract_method_code(lines_before, mb.start_line, mb.end_line)
                    fixed_code = extract_method_code(lines_after,  ma.start_line, ma.end_line)

                    if vuln_code.strip() != fixed_code.strip():
                        positives.append((vuln_code,  True))   # vulnerable label
                        positives.append((fixed_code, False))  # fixed label
                        stats["changed_pairs"] += 1
                    else:
                        negatives.append((fixed_code, False))
                        stats["unchanged_methods"] += 1

    except Exception as e:
        stats["errors"] += 1

    max_neg     = stats["changed_pairs"] * NEG_SAMPLE_RATIO
    sampled_neg = (random.sample(negatives, min(len(negatives), max_neg))
                   if max_neg > 0 else [])

    return positives + sampled_neg, dict(stats)


print(" Helper functions defined")

In [ ]:

repo_to_commits = defaultdict(list)
for entry in cve_data:
    repo_to_commits[entry["repo_url"]].append(entry["commit_hash"])

repo_to_commits = {url: list(set(hashes))
                   for url, hashes in repo_to_commits.items()}

if MAX_REPOS is not None:
    repo_to_commits = dict(list(repo_to_commits.items())[:MAX_REPOS])
    print(f"  Limited to first {MAX_REPOS} repos")

print(f"\n Ready to process:")
print(f"   Repos    : {len(repo_to_commits)}")
print(f"   Commits  : {sum(len(v) for v in repo_to_commits.values())}")

In [ ]:

dataset   = []                 # final list of (code_str, label) tuples
agg_stats = defaultdict(int)   # aggregated diagnostic counts
lock      = threading.Lock()   # needed because multiple threads write to dataset

pbar = tqdm(total=len(repo_to_commits), desc="Repos", unit="repo")


def worker(repo_url: str, hashes: list):
    samples, stats = process_repo(repo_url, hashes)
    with lock:
        dataset.extend(samples)
        for k, v in stats.items():
            agg_stats[k] += v
        pbar.update(1)
        pbar.set_postfix({"samples": len(dataset)})


with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [
        executor.submit(worker, url, hashes)
        for url, hashes in repo_to_commits.items()
    ]
    for fut in as_completed(futures):
        try:
            fut.result()
        except Exception:
            pass

pbar.close()

print(f"""
╔══════════════════════════════════════╗
  Extraction complete!
  Repos processed    : {len(repo_to_commits)}
  Clone failures     : {agg_stats.get('clone_fail', 0)}
  Commits processed  : {agg_stats.get('commits_processed', 0)}
  Files processed    : {agg_stats.get('files_processed', 0)}
  Methods compared   : {agg_stats.get('methods_compared', 0)}
    Changed pairs    : {agg_stats.get('changed_pairs', 0)}
    Unchanged        : {agg_stats.get('unchanged_methods', 0)}
  Raw samples        : {len(dataset)}
    Vulnerable (True): {sum(1 for _, l in dataset if l)}
    Safe (False)     : {sum(1 for _, l in dataset if not l)}
╚══════════════════════════════════════╝


In [ ]:

seen   = set()
deduped = []

for code_str, label in dataset:
    h = hashlib.md5(code_str.strip().encode("utf-8", errors="ignore")).hexdigest()
    if h not in seen:
        seen.add(h)
        deduped.append((code_str, label))

dataset = deduped

print(f"After deduplication:")
print(f"  Total      : {len(dataset)}")
print(f"  Vulnerable : {sum(1 for _, l in dataset if l)}")
print(f"  Safe       : {sum(1 for _, l in dataset if not l)}")

In [ ]:
import json
import pandas as pd

jsonl_path = os.path.join(DRIVE_OUTPUT_DIR, "vuln_dataset.jsonl")

with open(jsonl_path, "w", encoding="utf-8") as f:
    for code_str, label in dataset:
        record = {
            "function":   code_str,
            "vulnerable": bool(label)   # True = vulnerable, False = safe
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f" Saved JSONL: {jsonl_path}")

csv_path = os.path.join(DRIVE_OUTPUT_DIR, "vuln_dataset.csv")
df = pd.DataFrame(dataset, columns=["function", "vulnerable"])
df.to_csv(csv_path, index=False)
print(f" Saved CSV : {csv_path}")

print(f"""
╔══════════════════════════════════════╗
  Dataset saved to Google Drive!
  Records    : {len(dataset)}
  Vulnerable : {df['vulnerable'].sum()}  ({df['vulnerable'].mean()*100:.1f}%)
  Safe       : {(~df['vulnerable']).sum()}  ({(~df['vulnerable']).mean()*100:.1f}%)
  Location   : {DRIVE_OUTPUT_DIR}
╚══════════════════════════════════════╝
